# Scripts de extracción de datos
- Osvaldo Ceballos O. , Andrés Jimenez B.

In [7]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

import os
cwd= os.getcwd()
print (cwd)

C:\Users\oaceb\PycharmProjects\AnalisisLogisitca\src


# 1.- Datos aduanas

### 2.1 Fuente Datos Abiertos Gobierno de Chile (Datos.gov.cl)
https://datos.gob.cl/organization/servicio_nacional_de_aduanas

In [8]:
!pip install ckanapi requests openpyxl

In [9]:
import urllib
from ckanapi import RemoteCKAN
import requests
import pandas as pd
import numpy as np

In [12]:
# URL del portal CKAN
ckan_url = 'https://datos.gob.cl/'

# Conexión al portal CKAN
ckan = RemoteCKAN(ckan_url)

# Realizar una búsqueda de datasets
query = '(DUS) del año 202'  #@param
results = ckan.action.package_search(q=query)

# Mostrar nombres y descripciones de los primeros datasets encontrados
for dataset in results['results']:
    print(f"Nombre: {dataset['title']}")
    print(f"Descripción: {dataset['notes']}")
    print(f"Id: {dataset['id']}\n")

Nombre: Registro de Exportaciones 2026
Descripción: Muestra el registro de ítems correspondientes a los Documentos únicos de Salida (DUS) del año 2026. Registros mensuales. Formato comprimido WinRar.
Id: f6a643e7-a2e4-48e4-866e-732c7ceb51f3

Nombre: Registros de Exportación 2018
Descripción: Muestra el registro de ítem correspondientes a los Documentos Únicos de Salida (DUS) del año 2018. Registros mensuales.
Id: c9ddc990-53d9-4e50-86f0-d4b71cc6c803

Nombre: Registro de Exportaciones 2024
Descripción: Muestra el registro de ítems correspondientes a los Documentos únicos de Salida (DUS) del año 2024. Registros mensuales. Formato comprimido  WinRar.
Id: 1545f888-3490-466b-b815-60a0cd02ad23

Nombre: Registro de Exportación 2023
Descripción: Muestra el registro de ítems correspondientes a los Documentos únicos de Salida (DUS) del año 2023. Registros mensuales. Formato comprimido  WinRar.
Id: d4f8a96f-c555-4f68-b0a9-261dbd34292d

Nombre: Registro de Exportación 2021
Descripción: Muestra el 

In [13]:
import os
import requests
from urllib.parse import urlparse, unquote

data_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data"))
os.makedirs(data_dir, exist_ok=True)

for dataset in results['results']:
    dataset_id = dataset['id']

    # Obtener detalles del dataset
    dataset_sub = ckan.action.package_show(id=dataset_id)

    # Mostrar información relevante
    print(f"Nombre: {dataset['title']}")
    print(f"Descripción: {dataset['notes']}")
    print(f"Recursos disponibles:")

    for resource in dataset_sub['resources']:
        resource_url = resource.get('url')
        resource_name = resource.get('name') or 'recurso'
        print(f"- {resource_name}: {resource.get('format')} ({resource_url})")

        if not resource_url:
            continue

        parsed_url = urlparse(resource_url)
        filename = os.path.basename(parsed_url.path) or f"{resource_name}"
        filename = unquote(filename)
        file_path = os.path.join(data_dir, filename)

        response = requests.get(resource_url, stream=True, timeout=120)
        response.raise_for_status()

        with open(file_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)



Nombre: Registro de Exportaciones 2026
Descripción: Muestra el registro de ítems correspondientes a los Documentos únicos de Salida (DUS) del año 2026. Registros mensuales. Formato comprimido WinRar.
Recursos disponibles:
- Metadata Exportaciones: xlsx (https://datos.gob.cl/dataset/f6a643e7-a2e4-48e4-866e-732c7ceb51f3/resource/e0a25239-af2f-49be-afc3-44be524d6e4e/download/descripcion-y-estructura-de-datos-.xlsx)
- Exportaciones enero 2026: rar (https://datos.gob.cl/dataset/f6a643e7-a2e4-48e4-866e-732c7ceb51f3/resource/e8075003-9f64-4e3a-aa7e-f689a7e056eb/download/exportaciones-enero-2026.rar)
- Exportaciones enero 2026 – Bultos: rar (https://datos.gob.cl/dataset/f6a643e7-a2e4-48e4-866e-732c7ceb51f3/resource/6dd1316e-fe99-4a62-80c2-d838dcb3695a/download/exportaciones-enero-2026-bultos.rar)
- Exportaciones enero 2026 – Documentos de Transporte: rar (https://datos.gob.cl/dataset/f6a643e7-a2e4-48e4-866e-732c7ceb51f3/resource/f15ca5c2-ee96-4d47-be9b-61ca677ab4d7/download/exportaciones-enero

Esta parte se ejecuta para extraer los archivos .rar y consolidar

In [ ]:
#Para colab
#!apt-get install unrar

In [2]:
import os
import tempfile
from pathlib import Path
import pandas as pd
import rarfile


In [6]:
aduanas_dir = Path(os.path.abspath(os.path.join(os.getcwd(), "..", "data", "aduanas")))
extract_dir = Path(tempfile.mkdtemp(prefix="extracted_files"))

# Specify the full path to the unrar.exe file
rarfile.UNRAR_TOOL = r"C:\Program Files\WinRAR\UnRAR.exe" #comentar si se trabaja con linux/unix

df_list = []

part01_files = sorted(aduanas_dir.glob("*.part01.rar"))

for rar_file in part01_files:
    with rarfile.RarFile(rar_file) as rar_archive:
        rar_archive.extractall(path=extract_dir)

txt_files = sorted(extract_dir.rglob("*.txt"))

for txt_file in txt_files:
    try:
        df_tmp = pd.read_csv(txt_file, sep=";", low_memory=False, encoding="latin-1")
    except Exception:
        df_tmp = pd.read_csv(txt_file, sep=",", low_memory=False, encoding="latin-1")
    df_tmp["archivo_origen"] = txt_file.name
    df_list.append(df_tmp)
    txt_file.unlink(missing_ok=True)

aduanas_df = pd.concat(df_list, ignore_index=True) if df_list else pd.DataFrame()

aduanas_df.to_csv(aduanas_dir / "aduanas_df.csv", sep= ';', index=False, encoding="utf-8")


ParserError: Error tokenizing data. C error: Expected 14 fields in line 65, saw 15


### 2.2 Banco Central de Chile
https://si3.bcentral.cl/Siete/es/Siete/API?respuesta=

In [ ]:
!pip install bcchapi

In [ ]:
import bcchapi

# Incluyendo credenciales explícitamente
# O bien llamando a un archivo
siete = bcchapi.Siete(file="credenciales.txt")

In [ ]:
siete.buscar("M1")

,seriesId,frequencyCode,spanishTitle,englishTitle,firstObservation,lastObservation,updatedAt,createdAt
0,F021.M1.PRO.N.CLP.0.M,MONTHLY,"Agregados monetarios nominales saldos, M1 (Mil...","Balances nominal monetary aggregates, M1",1986-01-01,2024-07-01,2024-08-07,2024-08-07
1,F021.M1.PRO.N.CLP.1.M,MONTHLY,"Agregados monetarios nominales saldos, M1 dese...","Balances nominal monetary aggregates, M1 seaso...",1986-01-01,2024-07-01,2024-08-07,2024-08-07
2,F021.M1.STO.N.CLP.0.M,MONTHLY,"M1; agregados monetarios nominales, promedios ...","M1; nominal monetary aggregates, averaged (bil...",1965-12-01,2024-07-01,2024-08-07,2024-08-07
3,F021.M1.STO.N.CLP.HIST.M,MONTHLY,"Agregados monetarios nominales promedios, M1, ...",Base monetaria y agregados monetarios privados...,1965-12-01,2014-12-01,2015-05-28,2015-02-09
4,F021.M1.STO.R.P96.0.M,MONTHLY,"Agregados monetarios promedios reales , M1","Average real monetary aggregates, M1",1986-01-01,2023-12-01,2024-01-08,2024-01-08
5,F021.M1.STO.R.P96.1.M,MONTHLY,"Agregados monetarios promedios reales , M1 des...","Average real monetary aggregates, seasonally a...",1986-01-01,2023-12-01,2024-01-08,2024-01-08
6,F021.M1.STO.R.P96R23.0.M,MONTHLY,"M1 : Stock : Real : Pesos del 1996, referencia...","M1: Stock: Real: 1996 pesos, reference 2023: N...",1986-01-01,2024-07-01,2024-08-08,2024-08-08
7,F021.M1.STO.R.P96R23.1.M,MONTHLY,"M1 : Stock : Real : Pesos del 1996, referencia...","M1: Stock: Real: 1996 pesos, reference 2023: S...",1986-01-01,2024-07-01,2024-08-08,2024-08-08
8,F021.M1.STO.N.CLP.0.D,DAILY,Base monetaria y agregados monetarios privados...,Components of M1 (billions of pesos). M1,2011-01-03,2024-07-31,2024-08-07,2024-08-07
9,F021.M1.STS.N.CLP.0.D,DAILY,Base monetaria y agregados monetarios privados...,Base monetaria y agregados monetarios privados...,1996-01-07,2024-08-07,2024-08-16,2024-08-16


In [ ]:
series_bc = siete.cuadro(
  series=["F032.IMC.IND.Z.Z.EP18.Z.Z.0.M", "G073.IPC.IND.2018.M","F021.M1.STO.N.CLP.0.M"],
  nombres = ["imacec", "ipc","M1"],
  desde="2010-01-01",
  hasta="2024-07-01",
  #variacion=12,
  frecuencia="M",
  observado={"imacec":"mean", "ipc":"last", "M1":"mean"}
)

In [ ]:
series_bc = series_bc.fillna(0)
series_bc.head(50)

,imacec,ipc,M1
2010-01-31,72.478647,76.458780,13780.600000
2010-02-28,69.784256,76.673627,13876.100000
2010-03-31,75.714061,76.738097,14204.700000
2010-04-30,77.836154,77.092780,14333.800000
2010-05-31,77.265080,77.370183,15026.700000
2010-06-30,76.186444,77.370465,15322.400000
2010-07-31,74.112635,77.867485,15145.300000
2010-08-31,77.296543,77.790935,15125.600000
2010-09-30,75.050155,78.102046,15648.800000
2010-10-31,80.061679,78.178624,15449.000000


## Precios de los Combustibles - CNE
http://energiaabierta.cl/categorias-estadistica/hidrocarburos/?_sft_etiquetas-estadistica=precio

In [4]:
import bz2
from io import TextIOWrapper

cne_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "data", "cne"))
bz2_files = sorted(
    [
        os.path.join(cne_dir, f)
        for f in os.listdir(cne_dir)
        if f.lower().endswith(".csv.bz2") or f.lower().endswith(".bz2")
    ]
)

cne_df_list = []
for file_path in bz2_files:
    with bz2.open(file_path, "rt", encoding="utf-8", errors="replace") as f:
        df_tmp = pd.read_csv(f, low_memory=False, sep = ';', decimal=',', encoding="latin-1")
    df_tmp["archivo_origen"] = os.path.basename(file_path)
    cne_df_list.append(df_tmp)

cne_df = pd.concat(cne_df_list, ignore_index=True) if cne_df_list else pd.DataFrame()

cne_df.head()


,id,razon_social,distribuidor,direccion_calle,direccion_numero,comuna,region,precio,fecha_actualizacion,combustible,latitud,longitud,archivo_origen,"codigo,razon_social,distribuidor,direccion,latitud,longitud,nom_comuna,nom_region,combustible,precio,unidad_cobro,atencion,fecha_actualizacion,hora_actualizacion,es_electrolinera,es_gasolinera"
0,co1312002,Sociedad Herrera Bravo Ltda,COPEC,Avenida Irarrázaval,5277,Ñuñoa,Metropolitana,759.0,2012-01-01,Gasolina 93,"-33,4540062109273","-70,57525992393493",2012.csv.bz2,NaN
1,pb630302,Distribuidora Dagnino Giacobbe y Cía. Ltda. 7...,Sin Bandera,Camino a Codegua Km,01,Chimbarongo,Gral. Bernardo O'Higgins,612.0,2012-12-13,Petroleo Diesel,"-34,71739081954427","-71,02941691875458",2012.csv.bz2,NaN
2,co120501,SOCIEDAD COMERCIAL IBAï¿½EZ Y NEGRON LTDA.,COPEC,"PANAMERICANA NORTE KM 1750, EX S. V",0,Pozo Almonte,Tarapacá,777.0,2012-12-06,Gasolina 95,"-21,09225805724834","-69,5928230881691",2012.csv.bz2,NaN
3,te1312301,Gilberto Zamorano Vega,SHELL,Holanda,2808,Providencia,Metropolitana,790.0,2012-12-06,Gasolina 97,"-33,44420837761395","-70,59720039367676",2012.csv.bz2,NaN
4,co1313201,PATRICIO REYES INFANTE Y CIA LTDA,COPEC,Av. Vitacura,6380,Vitacura,Metropolitana,873.0,2012-09-06,Gasolina 97,"-33,38974288164123","-70,57051241397858",2012.csv.bz2,NaN


In [5]:
cne_df.to_csv(os.path.join(cne_dir,'cne_df.csv'),
              sep = ';',
              index = False,
              encoding='utf-8')